# AmesClean
### Data Cleaning & Data Quality Audit for the Ames Housing Dataset

**Programme:** NeuroFive Internship, Week 1, Machine Learning (Regression Track)
**Dataset:** Kaggle, *House Prices - Advanced Regression Techniques* (`train.csv`, 1,460 houses, 79 explanatory features, target `SalePrice`)
**Goal:** Turn the raw Ames Housing data into a documented, cleaned version where every data-quality decision is justified.

| Stage | What happens | Output |
|---|---|---|
| 1. Load & profile | Shape, types, target check | Overview tables |
| 2. Missing-value audit | Count and percent per column, missingness pattern | Figures 2 and 3 |
| 3. Duplicate audit | Exact and near-duplicate records | Duplicate summary |
| 4. Missing-value treatment | Column-by-column strategy with justification | Decision table |
| 5. Type correction | Codes to categories, ordered quality scales | Re-typing log |
| 6. Outlier audit | IQR and z-score, flag instead of delete | Figures 4 and 5 |
| 7. Scorecard & report | Before/after summary, `DATA_QUALITY_REPORT.md` | Figure 1, cleaned CSV, report |

**Scope note:** scaling, encoding and train/test splitting belong to Week 2. `SalePrice` is never used in any cleaning decision.

## 1. Setup

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, FancyBboxPatch
from matplotlib.colors import ListedColormap
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

PROJECT = "AmesClean"

INK = "#0b0b0b"
INK2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
SURFACE = "#fcfcfb"
TILE = "#f3f2ee"
BLUE = "#2a78d6"
ORANGE = "#eb6834"
SEQ_LIGHT = "#cde2fb"
SEQ_DARK = "#184f95"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS,
    "axes.labelcolor": INK2,
    "axes.titlecolor": INK,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.titlepad": 14,
    "legend.frameon": False,
})

os.makedirs("figures", exist_ok=True)


def save_fig(fig, name):
    fig.savefig("figures/" + name, dpi=160, bbox_inches="tight", facecolor=SURFACE)
    plt.show()

## 2. Load & profile the raw data
Upload `train.csv` when Colab asks for it. The target `SalePrice` and the `Id` column are kept out of every cleaning decision.

In [ ]:
path = "train.csv"
if not os.path.exists(path):
    from google.colab import files
    uploaded = files.upload()
    path = list(uploaded.keys())[0]

raw = pd.read_csv(path)
df = raw.copy()

target = "SalePrice" if "SalePrice" in raw.columns else None
id_col = "Id" if "Id" in raw.columns else None
feature_cols = [c for c in raw.columns if c not in (target, id_col)]

print(PROJECT, "| raw shape:", raw.shape)
print("Feature columns:", len(feature_cols))
print("Target column:", target)
display(raw.head())
display(raw.dtypes.astype(str).value_counts().rename("columns").to_frame())

## 3. Missing-value audit
The full count and percent for every column is saved to `missing_value_report.csv`. In this dataset the literal text `NA` often means *the feature is absent* (no pool, no garage, no basement), so the first question for each column is whether the gap is a real category or a genuine recording gap.

In [ ]:
def missing_report(frame):
    out = pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_pct": (frame.isna().mean() * 100).round(2),
        "dtype": frame.dtypes.astype(str),
    })
    return out.sort_values("missing_count", ascending=False)


full_missing = missing_report(raw)
missing_before = full_missing[full_missing["missing_count"] > 0]
full_missing.to_csv("missing_value_report.csv")

none_reasons = {
    "Alley": "NA in the source means no alley access, which is a real category",
    "PoolQC": "NA means the house has no pool, so the quality rating does not apply",
    "Fence": "NA means the house has no fence",
    "MiscFeature": "NA means no miscellaneous feature such as a shed or tennis court",
    "FireplaceQu": "NA means the house has no fireplace",
    "GarageType": "NA means the house has no garage",
    "GarageFinish": "NA means the house has no garage",
    "GarageQual": "NA means the house has no garage",
    "GarageCond": "NA means the house has no garage",
    "BsmtQual": "NA means the house has no basement",
    "BsmtCond": "NA means the house has no basement",
    "BsmtExposure": "NA means the house has no basement",
    "BsmtFinType1": "NA means the house has no basement",
    "BsmtFinType2": "NA means the house has no basement",
    "MasVnrType": "Missing veneer type goes with missing veneer area, meaning no masonry veneer",
}
absent_feature_cols = set(none_reasons) | {"GarageYrBlt", "MasVnrArea"}

print("=== MISSING VALUE REPORT (RAW DATA) ===")
print(len(missing_before), "columns out of", raw.shape[1], "have missing values")
display(missing_before)

In [ ]:
plot_df = missing_before.sort_values("missing_pct")
bar_colors = [BLUE if c in absent_feature_cols else ORANGE for c in plot_df.index]

fig, ax = plt.subplots(figsize=(10, 0.42 * len(plot_df) + 2.2))
ax.barh(plot_df.index, plot_df["missing_pct"], color=bar_colors, height=0.62)
for y, v in enumerate(plot_df["missing_pct"]):
    ax.text(v + 1.2, y, f"{v:.1f}%", va="center", ha="left", fontsize=9, color=INK2)
ax.set_xlim(0, 115)
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.set_xlabel("Share of houses with the value missing (%)")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.spines["left"].set_visible(False)
ax.set_title("Most missing values mean the feature is absent, not that data was lost")
ax.legend(
    handles=[
        Patch(color=BLUE, label="Feature absent (NA is a real category)"),
        Patch(color=ORANGE, label="Genuine recording gap"),
    ],
    loc="lower right",
)
save_fig(fig, "fig_02_missing_values.png")

In [ ]:
pattern_cols = list(missing_before.index)
mat = raw[pattern_cols].isna().to_numpy().T
order = np.argsort(-mat.sum(axis=0), kind="stable")
mat = mat[:, order]

fig, ax = plt.subplots(figsize=(10, 0.34 * len(pattern_cols) + 2.2))
ax.imshow(mat, aspect="auto", interpolation="nearest", cmap=ListedColormap([SEQ_LIGHT, SEQ_DARK]))
ax.set_yticks(range(len(pattern_cols)))
ax.set_yticklabels(pattern_cols, fontsize=8.5)
ax.set_xticks([])
ax.set_xlabel(f"{mat.shape[1]:,} houses, sorted by number of missing fields")
ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Missing values arrive in blocks, not at random")
ax.legend(
    handles=[Patch(color=SEQ_DARK, label="Missing"), Patch(color=SEQ_LIGHT, label="Present")],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.06),
    ncol=2,
)
save_fig(fig, "fig_03_missing_pattern.png")

## 4. Duplicate audit
Three levels are checked: exact duplicate records, duplicates ignoring `Id`, and near-duplicates (rows that differ in at most two of the feature columns). Only exact duplicate records are removed, because rows that merely look alike can be genuinely different houses.

In [ ]:
cols_no_id = [c for c in raw.columns if c != id_col]
exact_dups = int(raw.duplicated().sum())
dups_ignoring_id = int(raw.duplicated(subset=cols_no_id).sum())
dups_features_only = int(raw.duplicated(subset=feature_cols).sum())

near_keys = [c for c in ["Neighborhood", "YearBuilt", "LotArea", "GrLivArea", "OverallQual", "TotalBsmtSF"] if c in raw.columns]
near_key_mask = raw.duplicated(subset=near_keys, keep=False)
near_key_rows = raw[near_key_mask].sort_values(near_keys)

codes = np.column_stack([pd.factorize(raw[c])[0] for c in feature_cols])
near_pairs = []
for i in range(len(codes) - 1):
    diff = (codes[i + 1:] != codes[i]).sum(axis=1)
    for h in np.where(diff <= 2)[0]:
        near_pairs.append((int(i), int(i + 1 + h), int(diff[h])))

print("=== DUPLICATE CHECK ===")
print("Exact duplicate rows:", exact_dups)
print("Duplicate rows ignoring Id:", dups_ignoring_id)
print("Duplicate rows on features only:", dups_features_only)
print("Rows sharing key columns", near_keys, ":", len(near_key_rows))
print("Near-duplicate pairs (differ in at most 2 feature columns):", len(near_pairs))
if len(near_pairs) > 0:
    display(pd.DataFrame(near_pairs, columns=["row_a", "row_b", "columns_differing"]).head(20))
if len(near_key_rows) > 0:
    display(near_key_rows.head(20))

dup_mask = raw.duplicated(subset=cols_no_id)
rows_removed = int(dup_mask.sum())
df = df.loc[~dup_mask].copy()
print("Rows removed as exact duplicates:", rows_removed)

## 5. Missing-value treatment
Each strategy follows the *meaning* of the gap. Nothing is filled with a blanket column mean, and no row or column is dropped for missing values.

| Situation | Strategy |
|---|---|
| NA means the feature is absent (pool, garage, basement, fence, alley, fireplace, veneer) | Impute with the category `None` |
| Value exists in reality but was not recorded (`LotFrontage`) | Neighborhood median plus an indicator column |
| Year of a garage that does not exist (`GarageYrBlt`) | `YearBuilt` plus an indicator column |
| Area of a veneer that does not exist (`MasVnrArea`) | Fill with 0 |
| One-off gap in a standard system (`Electrical`) | Most frequent category |

In [ ]:
decisions = []

for c in [c for c in none_reasons if c in df.columns]:
    n = int(df[c].isna().sum())
    df[c] = df[c].fillna("None")
    decisions.append((c, n, "Impute with category None", none_reasons[c]))

if "LotFrontage" in df.columns:
    n = int(df["LotFrontage"].isna().sum())
    df["LotFrontage_was_missing"] = df["LotFrontage"].isna().astype(int)
    if "Neighborhood" in df.columns:
        df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(lambda s: s.fillna(s.median()))
    df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
    decisions.append((
        "LotFrontage", n,
        "Impute with neighborhood median plus indicator column LotFrontage_was_missing",
        "Every house touches a street so the value exists but was not recorded; lots in the same neighborhood have similar frontage, and the indicator keeps the imputed rows visible",
    ))

if "GarageYrBlt" in df.columns:
    n = int(df["GarageYrBlt"].isna().sum())
    df["GarageYrBlt_was_missing"] = df["GarageYrBlt"].isna().astype(int)
    base = df["YearBuilt"] if "YearBuilt" in df.columns else pd.Series(0, index=df.index)
    df["GarageYrBlt"] = df["GarageYrBlt"].fillna(base)
    decisions.append((
        "GarageYrBlt", n,
        "Impute with YearBuilt plus indicator column GarageYrBlt_was_missing",
        "Missing means no garage so no year exists; a mean year would invent a false date, YearBuilt keeps the column on a sensible scale and the indicator preserves the no garage signal",
    ))

if "MasVnrArea" in df.columns:
    n = int(df["MasVnrArea"].isna().sum())
    df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
    decisions.append((
        "MasVnrArea", n,
        "Impute with value 0",
        "Missing veneer area means no masonry veneer, so the true area is zero",
    ))

if "Electrical" in df.columns:
    n = int(df["Electrical"].isna().sum())
    if n > 0:
        mode_value = df["Electrical"].mode().iloc[0]
        df["Electrical"] = df["Electrical"].fillna(mode_value)
        decisions.append((
            "Electrical", n,
            "Impute with most frequent category " + str(mode_value),
            "Only a handful of rows are affected and every house has an electrical system, so the dominant category is the safest estimate",
        ))

leftover_numeric = [c for c in df.select_dtypes(include="number").columns if c != target and df[c].isna().any()]
for c in leftover_numeric:
    n = int(df[c].isna().sum())
    df[c + "_was_missing"] = df[c].isna().astype(int)
    df[c] = df[c].fillna(df[c].median())
    decisions.append((c, n, "Impute with median plus indicator column", "Skewed numeric column so the median is more robust than the mean, and the indicator keeps imputed rows visible"))

leftover_categorical = [c for c in df.select_dtypes(exclude="number").columns if df[c].isna().any()]
for c in leftover_categorical:
    n = int(df[c].isna().sum())
    df[c] = df[c].fillna(df[c].mode().iloc[0])
    decisions.append((c, n, "Impute with most frequent category", "Few missing values and no evidence that the gap carries meaning"))

decisions_df = pd.DataFrame(decisions, columns=["column", "missing_filled", "strategy", "justification"])
print("=== MISSING VALUE DECISIONS ===")
display(decisions_df[["column", "missing_filled", "strategy"]])

remaining = df.drop(columns=[target] if target else []).isna().sum().sum()
print("Missing values left in feature columns:", int(remaining))

## 6. Type correction
Numeric-looking codes become categories, quality ratings become ordered categories (`Po < Fa < TA < Gd < Ex`), and text columns become the `category` dtype.

In [ ]:
retype_log = []

for c in [c for c in ["MSSubClass", "MoSold"] if c in df.columns]:
    old_dtype = str(df[c].dtype)
    df[c] = df[c].astype(str)
    retype_log.append((c, old_dtype, "category", "Numeric looking code or month number with no arithmetic meaning"))

qual_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
qual_cols = [c for c in ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"] if c in df.columns]
for c in qual_cols:
    old_dtype = str(df[c].dtype)
    before_nan = int(df[c].isna().sum())
    df[c] = pd.Categorical(df[c], categories=qual_order, ordered=True)
    after_nan = int(df[c].isna().sum())
    retype_log.append((c, old_dtype, "ordered category", "Quality ratings have a natural order from Po to Ex"))
    if after_nan != before_nan:
        print("Warning: unexpected labels in", c)

if "GarageYrBlt" in df.columns and not df["GarageYrBlt"].isna().any():
    df["GarageYrBlt"] = df["GarageYrBlt"].astype(int)
    retype_log.append(("GarageYrBlt", "float64", "int64", "A year is a whole number"))

text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and not isinstance(df[c].dtype, pd.CategoricalDtype)]
for c in text_cols:
    old_dtype = str(df[c].dtype)
    df[c] = df[c].astype("category")
    retype_log.append((c, old_dtype, "category", "Text column with a fixed set of labels"))

retype_df = pd.DataFrame(retype_log, columns=["column", "from", "to", "reason"])
print("=== TYPE CORRECTIONS ===")
print(len(retype_df), "columns re-typed")
display(retype_df[["column", "from", "to"]])
display(df.dtypes.astype(str).value_counts().rename("columns").to_frame())

## 7. Logical consistency checks
Impossible combinations (a house built after it was sold, a remodel before construction, negative areas) would indicate real data errors rather than unusual-but-valid values.

In [ ]:
consistency = {}

if {"YearRemodAdd", "YearBuilt"} <= set(df.columns):
    consistency["YearRemodAdd earlier than YearBuilt"] = int((df["YearRemodAdd"] < df["YearBuilt"]).sum())
if {"YearBuilt", "YrSold"} <= set(df.columns):
    consistency["YearBuilt later than YrSold"] = int((df["YearBuilt"] > df["YrSold"]).sum())
if {"GarageYrBlt", "YrSold", "GarageYrBlt_was_missing"} <= set(df.columns):
    consistency["GarageYrBlt later than YrSold"] = int(((df["GarageYrBlt"] > df["YrSold"]) & (df["GarageYrBlt_was_missing"] == 0)).sum())
numeric_only = [c for c in df.select_dtypes(include="number").columns if c != target]
consistency["Negative values in numeric columns"] = int((df[numeric_only] < 0).sum().sum())

print("=== LOGICAL CONSISTENCY CHECKS ===")
for k, v in consistency.items():
    print(k, ":", v)

top_share = {}
for c in feature_cols:
    if c in df.columns:
        top_share[c] = float(df[c].astype(str).value_counts(normalize=True).iloc[0])
near_constant = {c: round(v * 100, 2) for c, v in top_share.items() if v >= 0.99}
print("Near-constant columns (99 percent or more one value):", near_constant)

## 8. Outlier audit
Outliers are detected with two rules: the **IQR rule** (outside Q1 - 1.5 x IQR or Q3 + 1.5 x IQR) and the **z-score rule** (absolute z above 3). Each column then gets a decision on whether the extreme values look like data errors or genuine extremes. Nothing is capped or deleted; every column gets a `*_outlier_flag` indicator so the decision can be revisited in Week 2.

In [ ]:
outlier_candidates = [c for c in ["GrLivArea", "LotArea", "LotFrontage", "TotalBsmtSF", "MasVnrArea", "1stFlrSF"] if c in df.columns]

outlier_notes = {
    "GrLivArea": ("Genuine extreme value, flag and keep", "Very large living areas are real big houses, not typing mistakes; the value agrees with the floor area columns where those are available"),
    "LotArea": ("Genuine extreme value, flag and keep", "Very large lots are plausible for rural or acreage properties and lot size is naturally right skewed"),
    "LotFrontage": ("Possible data issue, flag and keep for review", "Extreme frontage is unusual compared with lot area and part of it was imputed, so it is flagged rather than removed"),
    "TotalBsmtSF": ("Genuine extreme value, flag and keep", "A very large basement matches a very large house and is physically possible"),
    "MasVnrArea": ("Genuine extreme value, flag and keep", "Most houses have zero veneer so any real veneer area looks extreme by IQR, which is a property of the distribution not an error"),
    "1stFlrSF": ("Genuine extreme value, flag and keep", "Large first floor area goes with very large houses and is physically plausible"),
}

outlier_rows = []
for c in outlier_candidates:
    s = df[c].astype(float)
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_mask = (s < low) | (s > high)
    z = (s - s.mean()) / s.std()
    z_mask = z.abs() > 3
    df[c + "_outlier_flag"] = iqr_mask.astype(int)
    outlier_rows.append({
        "column": c,
        "min": round(float(s.min()), 2),
        "max": round(float(s.max()), 2),
        "lower_fence": round(float(low), 2),
        "upper_fence": round(float(high), 2),
        "iqr_outliers": int(iqr_mask.sum()),
        "zscore_outliers": int(z_mask.sum()),
        "decision": outlier_notes.get(c, ("Flag and keep", "Not a clear data error"))[0],
        "justification": outlier_notes.get(c, ("Flag and keep", "Not a clear data error"))[1],
    })

outlier_df = pd.DataFrame(outlier_rows)
print("=== OUTLIER ANALYSIS ===")
display(outlier_df[["column", "min", "max", "lower_fence", "upper_fence", "iqr_outliers", "zscore_outliers", "decision"]])

gr_check = None
if {"GrLivArea", "1stFlrSF", "2ndFlrSF", "LowQualFinSF"} <= set(df.columns):
    gr_mask = df["GrLivArea_outlier_flag"] == 1
    gr_total = int(gr_mask.sum())
    gr_ok = int((df.loc[gr_mask, "GrLivArea"] == df.loc[gr_mask, ["1stFlrSF", "2ndFlrSF", "LowQualFinSF"]].sum(axis=1)).sum())
    gr_check = (gr_ok, gr_total)
    print("GrLivArea outliers matching floor area sum:", gr_ok, "of", gr_total)

In [ ]:
panel_cols = list(outlier_df["column"])[:6]
n_rows = 2 if len(panel_cols) > 3 else 1
fig, axes = plt.subplots(n_rows, 3, figsize=(12, 3.3 * n_rows + 0.9), squeeze=False)
flat_axes = axes.ravel()

for ax, c in zip(flat_axes, panel_cols):
    info = outlier_df[outlier_df["column"] == c].iloc[0]
    ax.hist(df[c], bins=45, color=BLUE, edgecolor=SURFACE, linewidth=0.6, log=True)
    for fence in (info["lower_fence"], info["upper_fence"]):
        if df[c].min() <= fence <= df[c].max():
            ax.axvline(fence, color=INK2, linestyle="--", linewidth=1.2)
    ax.set_title(f"{c}  |  {int(info['iqr_outliers'])} flagged", fontsize=11)
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    ax.tick_params(length=0, labelsize=8.5)
    ax.spines["left"].set_visible(False)

for ax in flat_axes[len(panel_cols):]:
    ax.axis("off")

for ax in axes[:, 0]:
    ax.set_ylabel("Houses (log scale)")

fig.suptitle("Extreme values sit in the long right tail of each size column", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.text(0.01, 0.0, "Dashed lines mark the IQR fences (Q1 - 1.5 x IQR and Q3 + 1.5 x IQR). Flagged values are kept, not deleted. The vertical axis is logarithmic so single extreme houses stay visible.", fontsize=9, color=INK2, ha="left")
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
save_fig(fig, "fig_04_outlier_distributions.png")

In [ ]:
cmp_df = outlier_df.sort_values("iqr_outliers")
ypos = np.arange(len(cmp_df))
bar_h = 0.34

fig, ax = plt.subplots(figsize=(10, 0.95 * len(cmp_df) + 2.2))
ax.barh(ypos + bar_h / 2, cmp_df["iqr_outliers"], height=bar_h, color=BLUE, edgecolor=SURFACE, linewidth=1)
ax.barh(ypos - bar_h / 2, cmp_df["zscore_outliers"], height=bar_h, color=ORANGE, edgecolor=SURFACE, linewidth=1)
top = max(cmp_df["iqr_outliers"].max(), cmp_df["zscore_outliers"].max(), 1)
for y, a, b in zip(ypos, cmp_df["iqr_outliers"], cmp_df["zscore_outliers"]):
    ax.text(a + top * 0.015, y + bar_h / 2, str(int(a)), va="center", ha="left", fontsize=9, color=INK2)
    ax.text(b + top * 0.015, y - bar_h / 2, str(int(b)), va="center", ha="left", fontsize=9, color=INK2)
ax.set_yticks(ypos)
ax.set_yticklabels(cmp_df["column"])
ax.set_xlim(0, top * 1.12)
ax.set_xlabel("Number of values flagged")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.spines["left"].set_visible(False)
ax.set_title("The IQR rule flags more values than the z-score rule on skewed columns")
ax.legend(
    handles=[Patch(color=BLUE, label="IQR rule (1.5 x IQR)"), Patch(color=ORANGE, label="Z-score rule (|z| > 3)")],
    loc="lower right",
)
save_fig(fig, "fig_05_outlier_methods.png")

## 9. Validation & quality scorecard
Final checks: the target must be untouched, no feature column may still contain missing values, and the before/after numbers are summarised in one scorecard.

In [ ]:
if target:
    target_untouched = bool(df[target].equals(raw.loc[df.index, target]))
    print("SalePrice unchanged after cleaning:", target_untouched)
else:
    target_untouched = None
    print("No target column in this file")

missing_cells_before = int(raw.isna().sum().sum())
missing_cells_after = int(df.drop(columns=[target] if target else []).isna().sum().sum())
flag_cols = [c for c in df.columns if c.endswith("_outlier_flag")]
flagged_values = int(sum(df[c].sum() for c in flag_cols))

print("Raw shape:", raw.shape, "| Cleaned shape:", df.shape)
print("Missing cells:", missing_cells_before, "->", missing_cells_after)
print("Outlier flags raised:", flagged_values)

df.to_csv("ames_cleaned.csv", index=False)
print("Saved ames_cleaned.csv")

In [ ]:
tiles = [
    (f"{df.shape[0]:,}", "houses kept", f"{rows_removed} duplicate rows removed"),
    (f"{missing_cells_before:,} \u2192 {missing_cells_after}", "missing cells", f"{len(missing_before)} columns treated by meaning"),
    (str(len(retype_df)), "columns re-typed", "codes, scales, categories"),
    (f"{flagged_values:,}", "outlier values flagged", "none deleted or capped"),
]

fig, ax = plt.subplots(figsize=(12, 2.9))
ax.set_xlim(0, len(tiles))
ax.set_ylim(0, 1)
ax.axis("off")
for i, (value, label, note) in enumerate(tiles):
    ax.add_patch(FancyBboxPatch((i + 0.04, 0.06), 0.92, 0.7, boxstyle="round,pad=0,rounding_size=0.04", facecolor=TILE, edgecolor=GRID, linewidth=1))
    ax.text(i + 0.5, 0.54, value, ha="center", va="center", fontsize=25, fontweight="bold", color=INK)
    ax.text(i + 0.5, 0.33, label, ha="center", va="center", fontsize=11.5, color=INK2)
    ax.text(i + 0.5, 0.17, note, ha="center", va="center", fontsize=9, color=MUTED)
ax.text(0.04, 0.93, PROJECT + ": data quality scorecard", ha="left", va="center", fontsize=14, fontweight="bold", color=INK)
save_fig(fig, "fig_01_scorecard.png")

## 10. Export the data quality report
`DATA_QUALITY_REPORT.md` documents every decision and why, and embeds the figures from the `figures/` folder.

In [ ]:
def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for r in rows:
        lines.append("| " + " | ".join(str(x).replace("|", "/").replace("\n", " ") for x in r) + " |")
    return "\n".join(lines)


def fig_ref(caption, name):
    return "![" + caption + "](figures/" + name + ")"


today = pd.Timestamp.today().strftime("%d %B %Y")

L = []
L.append("# " + PROJECT + ": Data Quality Report")
L.append("")
L.append("**Ames Housing (House Prices, Regression Track)** | NeuroFive Internship, Week 1, Machine Learning | " + today)
L.append("")
L.append(fig_ref("Data quality scorecard", "fig_01_scorecard.png"))
L.append("")
L.append("## Executive summary")
L.append("")
L.append("- The raw file has " + str(raw.shape[0]) + " houses and " + str(raw.shape[1]) + " columns; " + str(len(missing_before)) + " columns contain missing values (" + format(missing_cells_before, ",") + " missing cells in total).")
L.append("- Most gaps are meaningful: the feature does not exist (no pool, garage, basement, fence, alley, fireplace). They became the category None instead of being filled with a mean or dropped.")
L.append("- Only LotFrontage and Electrical look like genuine recording gaps and were imputed with a neighborhood median and the most frequent category.")
L.append("- Exact duplicate rows: " + str(exact_dups) + ". Near-duplicate pairs reviewed: " + str(len(near_pairs)) + ". Rows removed: " + str(rows_removed) + ".")
L.append("- " + str(len(outlier_df)) + " numeric columns were screened for outliers with the IQR rule and the z-score rule. Extreme values were flagged, not deleted.")
L.append("- " + str(len(retype_df)) + " columns were re-typed (codes to categories, quality scales to ordered categories).")
L.append("- The target " + str(target) + " was not used in any cleaning decision and was verified unchanged: " + str(target_untouched) + ".")
L.append("")
L.append("## 1. Dataset overview")
L.append("")
L.append("- Source: Kaggle, House Prices - Advanced Regression Techniques (Ames Housing), train.csv")
L.append("- Raw shape: " + str(raw.shape[0]) + " rows x " + str(raw.shape[1]) + " columns")
L.append("- Cleaned shape: " + str(df.shape[0]) + " rows x " + str(df.shape[1]) + " columns (the extra columns are missing-value indicators and outlier flags)")
L.append("- Target variable: " + str(target) + " (regression problem)")
L.append("- Scaling and encoding are intentionally not done here, they belong to the modeling stage in Week 2")
L.append("")
L.append("## 2. Missing-value report")
L.append("")
L.append("pandas reads the literal text NA as a missing value, and in this dataset NA very often means the feature is absent, for example no pool or no garage. The full count and percent for every column is saved in missing_value_report.csv.")
L.append("")
L.append(fig_ref("Missing values by column", "fig_02_missing_values.png"))
L.append("")
L.append(md_table(["Column", "Missing count", "Missing %", "Dtype"], [(r.Index, int(r.missing_count), r.missing_pct, r.dtype) for r in missing_before.itertuples()]))
L.append("")
L.append(fig_ref("Missingness pattern across houses", "fig_03_missing_pattern.png"))
L.append("")
L.append("Conceptually, most of these gaps are not missing completely at random. The garage, basement, pool, fence, alley and fireplace columns are missing because the house does not have that feature, which is a not-at-random pattern that carries real information. Treating them as random gaps and filling with a mean would destroy that information. Only LotFrontage and Electrical look like true recording gaps.")
L.append("")
L.append("## 3. Missing-value decisions")
L.append("")
L.append(md_table(["Column", "Rows filled", "Strategy", "Why"], [(r.column, r.missing_filled, r.strategy, r.justification) for r in decisions_df.itertuples()]))
L.append("")
L.append("No rows were dropped for missing values, because dropping any row with a single missing value would throw away most of the dataset and bias it towards houses that have every feature. No column was dropped for the same reason: even PoolQC, which is almost entirely missing, is meaningful as the category None.")
L.append("")
L.append("## 4. Duplicate check")
L.append("")
L.append("- Exact duplicate rows: " + str(exact_dups))
L.append("- Duplicate rows when Id is ignored: " + str(dups_ignoring_id))
L.append("- Duplicate rows on feature columns only: " + str(dups_features_only))
L.append("- Rows sharing all of " + ", ".join(near_keys) + ": " + str(len(near_key_rows)))
L.append("- Near-duplicate pairs (differing in at most 2 of " + str(len(feature_cols)) + " feature columns): " + str(len(near_pairs)))
L.append("- Rows removed: " + str(rows_removed))
L.append("")
L.append("Rows that only share a few key columns can be different houses, so they were reviewed but not removed. Only exact duplicate records are removed.")
L.append("")
L.append("## 5. Outlier analysis")
L.append("")
L.append("Outliers were detected with the IQR rule (1.5 x IQR fences) and with the z-score rule (absolute z above 3). Nothing was capped or deleted. Each column got a 0/1 flag column named column_outlier_flag so the decision can be revisited later.")
L.append("")
L.append(fig_ref("Outlier distributions", "fig_04_outlier_distributions.png"))
L.append("")
L.append(fig_ref("IQR rule versus z-score rule", "fig_05_outlier_methods.png"))
L.append("")
L.append(md_table(["Column", "Min", "Max", "Lower fence", "Upper fence", "IQR outliers", "Z-score outliers", "Decision", "Why"], [(r.column, r.min, r.max, r.lower_fence, r.upper_fence, r.iqr_outliers, r.zscore_outliers, r.decision, r.justification) for r in outlier_df.itertuples()]))
if gr_check is not None:
    L.append("")
    L.append("Cross-check: " + str(gr_check[0]) + " of " + str(gr_check[1]) + " flagged GrLivArea rows equal the sum of 1stFlrSF, 2ndFlrSF and LowQualFinSF, which supports treating them as genuine large houses rather than entry errors.")
L.append("")
L.append("## 6. Type corrections")
L.append("")
L.append(md_table(["Column", "From", "To", "Reason"], [(r.column, r._2, r.to, r.reason) for r in retype_df.itertuples()]))
L.append("")
L.append("## 7. Logical consistency checks")
L.append("")
for k, v in consistency.items():
    L.append("- " + k + ": " + str(v))
L.append("- Near-constant columns (99 percent or more one value): " + (", ".join(k + " (" + str(v) + "%)" for k, v in near_constant.items()) if near_constant else "none"))
L.append("")
L.append("Near-constant columns carry almost no information but are kept for now; whether to drop them is a modeling-stage decision.")
L.append("")
L.append("## 8. Leakage risk (flagged, not fixed yet)")
L.append("")
L.append("The medians, the most frequent categories and the IQR fences above were computed on the full raw dataset because this task is a cleaning exercise before any train/test split. Once splitting begins in Week 2, every such statistic must be computed from the training data only and then applied to the test data, otherwise information from the test set leaks into training.")
L.append("")
L.append("## 9. Deliverables")
L.append("")
L.append("- ames_cleaned.csv: cleaned dataset")
L.append("- missing_value_report.csv: count and percent of missing values for every column")
L.append("- DATA_QUALITY_REPORT.md: this report")
L.append("- figures/: the five charts used in this report")

report_text = "\n".join(L) + "\n"
with open("DATA_QUALITY_REPORT.md", "w", encoding="utf-8") as f:
    f.write(report_text)
print("Saved DATA_QUALITY_REPORT.md")
print(len(report_text), "characters")

## 11. Package and download
Everything is bundled into one zip file so it can be submitted as a single deliverable.

In [ ]:
package = PROJECT + "_Submission"
os.makedirs(package, exist_ok=True)
for name in ["ames_cleaned.csv", "missing_value_report.csv", "DATA_QUALITY_REPORT.md"]:
    shutil.copy(name, package)
shutil.copytree("figures", package + "/figures", dirs_exist_ok=True)
shutil.make_archive(package, "zip", package)
print("Created", package + ".zip")

try:
    from google.colab import files
    files.download(package + ".zip")
except ImportError:
    print("Not running in Colab, files are saved in the current folder")